# 지표를 바꾸면 순위가 바뀌나 — micro p@5 / Macro p@5 / Macro mAP@5 실측

지표 조사(`지표_조사.md`)를 하다가 제일 먼저 확인해야 할 게 이거라고 생각했다:
**대표 지표를 바꾸면 방법들의 순위가 바뀌는가?** 바뀌면 지표 선택이 결과를 가르는 중대사고,
안 바뀌면 "무엇이 우리 이야기를 제일 잘 설명하나"로 편하게 고르면 되는 문제가 된다.

지금 리더보드에 있는 4개 방법을 세 지표로 전부 다시 쟀다. 누수 차단 규칙(자기 자신 +
같은 group 제외)은 채점기 score.py 와 동일하고, micro p@5 가 리더보드 숫자와 일치하는지
assert 로 검증해서 "같은 규칙으로 쟀다"를 보장했다.

- **micro p@5**: 지금 리더보드. 질의 2,522장 전부 평균 -> 큰 클래스가 지배
- **Macro p@5**: 클래스별 평균 먼저, 그 20개를 다시 평균 -> 클래스마다 한 표
- **Macro mAP@5**: 상위 5장 안에서 정답을 앞에 놨는지(순서)까지 반영한 AP@5 의 클래스 평균

이 노트북은 로컬(GPU)에서 실행한 출력을 그대로 담았다. 코랩에서 다시 돌리려면
코퍼스(daypack_v2)와 kit 이 같은 폴더에 풀려 있어야 한다.


In [1]:
import os
os.environ["USE_TF"] = "0"              # transformers 가 텐서플로를 끌어와 내는 잡음 방지 (torch 만 쓴다)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import sys, csv, json, importlib.util
import numpy as np
from PIL import Image

# 코퍼스와 채점기 위치를 찾는다 (로컬이면 현재 폴더, 코랩이면 /content)
ROOT = next(p for p in ['.', '..', '/content'] if os.path.exists(os.path.join(p, 'daypack_v2', 'meta.csv')))
KIT = os.path.join(ROOT, 'kit')
PACK = os.path.join(ROOT, 'daypack_v2')
K = 5

rows = list(csv.DictReader(open(os.path.join(PACK, 'meta.csv'), encoding='utf-8')))
imgs = [Image.open(os.path.join(PACK, r['file'])).convert('RGB') for r in rows]
style = np.array([r['style'] for r in rows])
group = np.array([r['group'] or '' for r in rows])
print(f"코퍼스 {len(imgs)}장 / {len(set(style))}클래스")


코퍼스 2522장 / 20클래스


In [2]:
def load_encode(path):
    # 채점기와 같은 방식으로 encode() 함수를 불러온다
    spec = importlib.util.spec_from_file_location('enc_' + os.path.basename(path), path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = mod
    spec.loader.exec_module(mod)
    return mod.encode

def metrics(E):
    # score.py 의 precision_at_k 와 같은 누수 규칙 + 순위를 보존해서 AP@5 까지 계산
    E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-8)
    N = len(E)
    p5, ap5 = np.zeros(N), np.zeros(N)
    for i in range(N):
        ok = (group != group[i]) | (group == '')   # 같은 원본/작가 묶음 제외 (내용 누수 차단)
        ok[i] = False                              # 자기 자신 제외
        sims = E @ E[i]
        sims[~ok] = -np.inf
        top = np.argpartition(-sims, K)[:K]
        top = top[np.argsort(-sims[top])]          # 유사도 순 = 순위
        rel = (style[top] == style[i]).astype(float)
        p5[i] = rel.mean()                         # p@5: 순서 안 봄
        cum = np.cumsum(rel)                       # AP@5: 정답 나온 순위마다 그 시점 정밀도를 더함
        ap5[i] = float((rel * cum / np.arange(1, K + 1)).sum() / K)
    return p5, ap5

def macro(v):
    # 클래스별 평균의 평균 (클래스마다 한 표)
    return float(np.mean([v[style == s].mean() for s in sorted(set(style))]))


In [3]:
METHODS = [
    ("CLIP 기준선", "encoders/clip_base.py"),
    ("민욱-흑백CLIP", "encoders/clip_gray.py"),
    ("민욱-패치셔플CLIP", "encoders/clip_patchshuffle.py"),
    ("Gram VGG19", "encoders/gram_vgg19.py"),
]

# 리더보드의 micro p@5 와 대조 -> 같은 규칙으로 쟀다는 검증
lb = {r["이름"]: float(r["p@5"]) for r in csv.DictReader(open(os.path.join(KIT, "leaderboard.csv"), encoding="utf-8"))
      if r["코퍼스"] == "daypack_v2"}
lb_map = {"CLIP 기준선": "CLIP 기준선 v2", "Gram VGG19": "Gram VGG19 v2"}

out = {}
print(f"{'방법':22s} {'micro p@5':>10s} {'Macro p@5':>10s} {'Macro mAP@5':>12s}")
for name, rel_path in METHODS:
    E = np.asarray(load_encode(os.path.join(KIT, rel_path))(imgs), dtype="float32")
    p5, ap5 = metrics(E)
    micro = float(p5.mean())
    board = lb.get(lb_map.get(name, name))
    if board is not None:
        assert abs(micro - board) < 5e-4, f"{name}: 리더보드와 불일치 (규칙 어긋남)"
    out[name] = {"micro_p5": round(micro, 4), "macro_p5": round(macro(p5), 4),
                 "macro_map5": round(macro(ap5), 4)}
    r = out[name]
    print(f"{name:22s} {r['micro_p5']:>10.4f} {r['macro_p5']:>10.4f} {r['macro_map5']:>12.4f}")

print()
for key in ["micro_p5", "macro_p5", "macro_map5"]:
    order = sorted(out, key=lambda n: -out[n][key])
    print(f"{key:11s} 순위: {' > '.join(order)}")


방법                      micro p@5  Macro p@5  Macro mAP@5


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


/home/gmw/anaconda3/envs/aiffel/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:96: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:309.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
/home/gmw/anaconda3/envs/aiffel/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:96: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:360.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


CLIP 기준선                   0.3986     0.4289       0.3536


민욱-흑백CLIP                  0.3711     0.3904       0.3172


민욱-패치셔플CLIP                0.4271     0.4401       0.3604


Gram VGG19                 0.4914     0.4774       0.4033

micro_p5    순위: Gram VGG19 > 민욱-패치셔플CLIP > CLIP 기준선 > 민욱-흑백CLIP
macro_p5    순위: Gram VGG19 > 민욱-패치셔플CLIP > CLIP 기준선 > 민욱-흑백CLIP
macro_map5  순위: Gram VGG19 > 민욱-패치셔플CLIP > CLIP 기준선 > 민욱-흑백CLIP


## 결론

**세 지표 모두 순위가 같다.** 지표를 바꿔도 지금까지의 결론(Gram > 패치셔플 > CLIP > 흑백)은
안 뒤집힌다. 그래서 대표 지표는 결과 걱정 없이 "우리 이야기를 제일 잘 설명하는 것"으로 고르면 된다.

회의에서 정하면 좋을 것:
1. 대표 지표 하나 (지금 p@5 유지 vs Macro mAP@5 로 교체) — 어느 쪽이든 순위는 같다
2. 병기 여부 — 리더보드에 두 지표를 같이 적어 순위 불변을 계속 확인하는 절충안
3. k=5 고정 확인 — 우리는 생성에 참조 5장을 쓰니까 k=5 가 서비스와 맞는다 (`지표_조사.md` 2장)

계산의 전제 하나: AP@5 의 분모를 5로 고정했는데, 모든 클래스에 정답 후보가 5장 이상이라는
전제다(우리 코퍼스 최소 클래스 24장이라 성립). 코퍼스가 바뀌면 다시 확인해야 한다.
